# Project 20 — Capstone: Compound Prioritization & Decision Under Uncertainty

**Scenario.** We screened $J$ compounds in a noisy assay with **unequal replication**. We must pick the single best compound to advance. This capstone integrates the whole workflow — a **hierarchical** model with partial pooling, full diagnostics + PPC, a **LOO** model comparison — and then crosses the finish line the other projects stop short of: it turns the posterior into an **explicit decision** via a cost-loss / expected-utility calculation.

**Key pitfall.** *Stopping at the posterior.* And, in the broken version, ranking by **raw (unpooled) means** — which falls for the small-sample **winner's curse**.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Each compound $j$ has a true effect $\theta_j$; we observe replicate assay readings $y_{j,i}\sim N(\theta_j,\sigma)$. The compounds are exchangeable draws $\theta_j\sim N(\mu,\tau)$. **Replication is unequal** — a few compounds have many replicates, several have only 2-3 — which is exactly what lets a noisy compound post a lucky-high raw mean. We know the truths and the TRUE best compound.

In [ ]:
from data.generate_data import generate
data = generate()
raw_pick = int(np.argmax(data['raw_means']))
print(f"J={data['j']} compounds; replicates={data['n_reps']}")
print(f"TRUE best = #{data['best_true']} (theta={data['theta_true'][data['best_true']]:.2f})")
print(f"RAW-mean winner = #{raw_pick} (raw={data['raw_means'][raw_pick]:.2f}, "
      f"n={data['n_reps'][raw_pick]}) <- the winner's-curse trap")

## Step 2 — Model specification (hierarchical, non-centred)

$$\mu\sim N(0,2),\;\tau\sim\text{HalfNormal}(1),\;\theta_j=\mu+\tau z_j,\;z_j\sim N(0,1),\;\sigma\sim\text{HalfNormal}(1),\;y_{j,i}\sim N(\theta_j,\sigma).$$

**Partial pooling** is the engine: compounds with few replicates are shrunk toward $\mu$ (their lucky-high raw means are pulled back), while well-replicated compounds barely move. We use the **non-centred** form ($\theta_j=\mu+\tau z_j$) to avoid the hierarchical funnel.

In [ ]:
from model import build_model, fit, fit_pooled, decision_table, posterior_theta
model = build_model(data)
model

## Step 3 — Prior predictive checks

We simulate compound screens from the priors. We want plausible spreads of effects — not all compounds identical (tau prior too tight) nor implausibly extreme (too loose).

In [ ]:
rng = np.random.default_rng(RNG)
fig, ax = plt.subplots(figsize=(6,3.5))
for _ in range(200):
    mu = rng.normal(0,2); tau = abs(rng.normal(0,1))
    ax.plot(rng.normal(mu, tau, size=data['j']), color='#4C72B0', alpha=0.05)
ax.set(xlabel='compound index', ylabel='theta drawn from prior',
       title='Prior predictive compound effects')
plt.tight_layout()

## Step 4 — Inference (NUTS)

Settings: `draws=600, tune=1000, chains=2, target_accept=0.95, cores=1`. The non-centred parameterisation plus a high `target_accept` keep the hierarchical geometry divergence-free.

In [ ]:
idata = fit(data, draws=600, tune=1000, chains=2, seed=101)

## Step 5 — Computational diagnostics

Check $\hat R$, ESS, and divergences (want 0 — the funnel is the usual culprit, tamed by non-centring). We also visualise the **shrinkage**: posterior $\theta_j$ means vs raw means; few-replicate compounds should be pulled toward $\mu$.

In [ ]:
print(az.summary(idata, var_names=['mu','tau','sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
th = posterior_theta(idata)
post_means = th.mean(axis=0)
fig, ax = plt.subplots(figsize=(6,3.8))
ax.scatter(data['raw_means'], post_means, c=data['n_reps'], cmap='viridis', zorder=3)
lims=[min(data['raw_means'].min(),post_means.min())-0.3,
      max(data['raw_means'].max(),post_means.max())+0.3]
ax.plot(lims, lims, 'k--', alpha=0.5, label='no shrinkage')
ax.set(xlabel='raw mean', ylabel='posterior theta mean',
       title='Shrinkage (colour = #replicates)')
ax.legend(); plt.colorbar(ax.collections[0], ax=ax, label='n_reps'); plt.tight_layout()

## Step 6 — Posterior predictive checks

We confirm the model reproduces the spread of the observed replicate data (`az.plot_ppc`). A good hierarchical fit captures both within- and between-compound variation.

In [ ]:
ax = az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

## Step 7 — Model comparison (hierarchical vs complete pooling) via LOO

Is the hierarchical structure earning its keep? We compare against a **complete-pooling** model (one shared effect) using LOO. When compounds genuinely differ, the hierarchical model should win (higher `elpd_loo`).

In [ ]:
idata_pooled = fit_pooled(data, draws=600, tune=1000, chains=2, seed=101)
cmp = az.compare({'hierarchical': idata, 'complete_pool': idata_pooled}, ic='loo')
print(cmp[['rank','elpd_loo','p_loo','dse']])

## Step 8 — DECISION (the point of the capstone)

A posterior is not a decision. We define a **utility**: advancing compound $j$ yields $U_j=\theta_j-\text{cost}$. From the posterior we compute, per compound, the **expected utility**, the **probability of being best**, and the **expected regret** $E[\max_k\theta_k-\theta_j]$. We recommend the compound that **minimises expected regret**. Watch how this can differ from picking the highest posterior mean — and how it overturns the raw-mean winner's curse.

In [ ]:
dec = decision_table(idata, cost=0.0)
import pandas as pd
tab = pd.DataFrame({'n_reps': data['n_reps'], 'raw_mean': data['raw_means'],
                    'exp_util': dec['exp_util'], 'P_best': dec['p_best'],
                    'exp_regret': dec['exp_regret']})
print(tab.round(3))
print()
print(f"true best          = #{data['best_true']}")
print(f"raw-mean winner    = #{int(np.argmax(data['raw_means']))} (winner's curse)")
print(f"posterior-mean pick= #{dec['argmax_posterior_mean']}")
print(f"MIN-REGRET pick    = #{dec['recommend_min_regret']}  <-- recommendation")

**Reading the table.** `P_best` spreads probability across several plausible winners — honesty the raw maximum hides. The min-expected-regret choice balances *being good* against *uncertainty about being best*. It corrects the winner's curse: the lucky low-$n$ raw winner is shrunk away, and a well-supported compound is advanced.

## Communication

We recommend advancing the min-expected-regret compound, and we report its probability of being best and the runner-up — so the team knows whether to advance one compound or carry two. See `summary_onepager.md` for the non-technical version. **This is the capstone lesson: the workflow ends in a decision, not a posterior.**